[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/08_attention_core.ipynb)

# 08. Attention core: MHA to MLA

동일한 작은 Q/K/V에서 single-head → MHA → MQA → GQA → latent-compressed attention으로 확장한다.

**반복 형식:** 바닐라 PyTorch 실행 → profiler로 ATen/CUDA 연산 확인 → 필요할 때만 작은 텐서로 수학적 전개를 펼친다.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("torch:", torch.__version__)


In [ ]:
from torch.profiler import profile, ProfilerActivity

def profile_call(name, fn, *args, **kwargs):
    activities = [ProfilerActivity.CPU]
    if torch.cuda.is_available():
        activities.append(ProfilerActivity.CUDA)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    with profile(
        activities=activities,
        record_shapes=True,
        profile_memory=True,
        with_stack=False,
    ) as prof:
        out = fn(*args, **kwargs)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    print(f"\n[{name}] top operators")
    sort_key = "self_cuda_time_total" if torch.cuda.is_available() else "self_cpu_time_total"
    print(prof.key_averages().table(sort_by=sort_key, row_limit=12))

    return out


## 1. Scaled dot-product attention

score → scale → softmax → value weighted sum을 explicit하게 본다.


In [ ]:
q = torch.tensor([[[[1., 0.], [0., 1.], [1., 1.]]]], device=device)
k = q.clone()
v = torch.tensor([[[[1., 2.], [3., 4.], [5., 6.]]]], device=device)

scores = q @ k.transpose(-2, -1) / math.sqrt(q.size(-1))
weights = scores.softmax(dim=-1)
out = weights @ v

print("scores:\n", scores)
print("weights:\n", weights)
print("output:\n", out)


In [ ]:
def explicit_sdpa(q_, k_, v_):
    scores_ = q_ @ k_.transpose(-2, -1)
    scores_ = scores_ / math.sqrt(q_.size(-1))
    return scores_.softmax(-1) @ v_

_ = profile_call("SDPA explicit", explicit_sdpa, q, k, v)
_ = profile_call("PyTorch SDPA", F.scaled_dot_product_attention, q, k, v)


## 2. Multi-head attention

head 축을 명시한다.


In [ ]:
B, H, T, Dh = 1, 2, 4, 4
q = torch.randn(B, H, T, Dh, device=device)
k = torch.randn_like(q)
v = torch.randn_like(q)

out = F.scaled_dot_product_attention(q, k, v)
print("MHA q/k/v:", q.shape, k.shape, v.shape)
print("MHA out:", out.shape)


In [ ]:
_ = profile_call("MHA SDPA", F.scaled_dot_product_attention, q, k, v)


## 3. MQA

모든 Q head가 하나의 K/V head를 공유한다.


In [ ]:
k_mqa = torch.randn(B, 1, T, Dh, device=device)
v_mqa = torch.randn(B, 1, T, Dh, device=device)

k_shared = k_mqa.expand(B, H, T, Dh)
v_shared = v_mqa.expand(B, H, T, Dh)

out_mqa = F.scaled_dot_product_attention(q, k_shared, v_shared)
print("stored K/V:", k_mqa.shape, v_mqa.shape)
print("logical K/V:", k_shared.shape, v_shared.shape)


In [ ]:
_ = profile_call("MQA", F.scaled_dot_product_attention, q, k_shared, v_shared)


## 4. GQA

Q head를 K/V group에 매핑한다.


In [ ]:
Hq, Hkv = 4, 2
qg = torch.randn(B, Hq, T, Dh, device=device)
kg = torch.randn(B, Hkv, T, Dh, device=device)
vg = torch.randn(B, Hkv, T, Dh, device=device)

repeat = Hq // Hkv
kg_expanded = kg.repeat_interleave(repeat, dim=1)
vg_expanded = vg.repeat_interleave(repeat, dim=1)

out_gqa = F.scaled_dot_product_attention(qg, kg_expanded, vg_expanded)
print("GQA stored K:", kg.shape, "expanded K:", kg_expanded.shape)


In [ ]:
_ = profile_call("GQA", F.scaled_dot_product_attention, qg, kg_expanded, vg_expanded)


## 5. MLA-style latent compression

K/V를 작은 latent로 압축한 뒤 각각 projection한다.


In [ ]:
B, T, D, R = 1, 4, 8, 3
x = torch.randn(B, T, D, device=device)

down = nn.Linear(D, R, bias=False).to(device)
k_up = nn.Linear(R, D, bias=False).to(device)
v_up = nn.Linear(R, D, bias=False).to(device)

latent = down(x)
k_mla = k_up(latent)
v_mla = v_up(latent)

print("x:", x.shape)
print("latent:", latent.shape)
print("reconstructed K/V:", k_mla.shape, v_mla.shape)


In [ ]:
_ = profile_call("MLA latent projections", lambda z: (k_up(down(z)), v_up(down(z))), x)


## References and provenance

**[8.1] Scaled dot-product / MHA**
- 출처: Vaswani et al., Attention Is All You Need
- 이 노트북에서 가져온 부분: attention baseline

**[8.2] MQA**
- 출처: Shazeer, Fast Transformer Decoding
- 이 노트북에서 가져온 부분: shared K/V heads

**[8.3] GQA**
- 출처: Ainslie et al., GQA
- 이 노트북에서 가져온 부분: grouped K/V heads

**[8.4] MLA**
- 출처: DeepSeek-V2 / DeepSeek-V3 technical papers
- 이 노트북에서 가져온 부분: latent compression of key/value representations
